# Notebook 1: What "learning" physically means (gradient descent from first principles)

This is step 1 of the ladder. Nothing here assumes you know machine learning. It uses only NumPy and a plot library, runs in a few seconds, and every number in the text under a cell is printed by the code above it.

**Why start here.** In the Qmaze and CartPole notebooks, the line `new = old + alpha * (target - old)` kept appearing, and Keras's `train_on_batch` was "nudge all the weights". Both are the same machine: **gradient descent**. By the end of this notebook you'll see exactly why that update works, where `alpha` comes from, and why it can fail.

### How these notebooks work
| Marker | What it is |
|---|---|
| `# WHAT / WHY / SAME AS / BECOMES / WATCH` | the same comment tags as before: the line in plain words, why it exists, the same idea elsewhere, what it turns into later, a trap |
| **PREDICT** cells | a cell with a blank variable. Type a guess *before* you run the next cell. Guessing first is what makes the answer stick |
| **Knob ledger** | at the end, a table of every knob we met, where it comes from, and what breaks at each extreme. It grows through the series |
| **Sandbox** | a slot where *your* idea competes against standard methods on the same ruler |

### The whole idea in three lines
1. **Parameters:** the adjustable numbers inside a model (a Q-table cell, a network weight, the slope of a line).
2. **Loss:** one number that says how wrong the current parameters are.
3. **Search:** a rule for changing the parameters so the loss goes down.

Gradient descent is one very good search rule: *measure which way is downhill, and take a step that way.*

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# WHAT: plot styling only (quiet grid, thin lines, fixed colors). Nothing here affects the math.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, MUTED, GRIDC = "#0b0b0b", "#52514e", "#e1e0d9"
BLUE_RAMP = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]       # light -> dark = early -> late
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlesize": 11, "axes.labelsize": 10, "font.size": 10,
    "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

def tag(ax, x, y, text, color, dx=4, dy=0, ha="left"):
    # WHAT: write a small label next to a line instead of making you decode a legend.
    ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points", color=color, fontsize=9, va="center", ha=ha)

print("ready")

ready


# Part 1: One dial, one number

> **You are here:** the smallest possible learning problem.
> **The setup:** one parameter `w`, and a loss that says how wrong `w` is. Pretend a machinist is setting a dial. They can't see the correct setting, only read an error gauge. The gauge reads 0 at the right setting and gets bigger the further off the dial is.

Here the error gauge is `(w - 3)^2`. The right answer is `w = 3`, **but the algorithm is never told that.** It may only read the gauge, and measure its slope.

**The slope** answers: *if I nudge `w` up a hair, does the error go up or down, and how fast?* You can measure it without any calculus by nudging and looking.

In [2]:
def loss(w):
    # WHAT: the "error gauge". 0 when w is exactly right (w = 3), bigger the further away w is.
    # WHY:  learning needs ONE number that says how wrong we are. This is the only thing the algorithm can read.
    # BECOMES: Part 3 uses the average of squared misses over many data points. In the Qmaze update it is (target - guess)^2.
    return (w - 3.0) ** 2

def slope_by_nudging(f, w, h=1e-5):
    # WHAT: measure the slope of f at w with NO calculus: nudge w up a hair, nudge it down a hair, see how f changes.
    # WHY:  "nudge and measure" works on ANY function you can evaluate, so it is always available as a check on a formula.
    return (f(w + h) - f(w - h)) / (2 * h)

w = 0.0
print(f"at w = {w}:  error = {loss(w):.3f}")
print(f"slope by nudging           = {slope_by_nudging(loss, w):+.4f}")
print(f"slope from the calculus rule 2(w - 3) = {2 * (w - 3.0):+.4f}   (the known rule for the slope of (w-3)^2)")
print()
print(f"nudge w up by 0.01: error goes {loss(w):.4f} -> {loss(w + 0.01):.4f}")

at w = 0.0:  error = 9.000
slope by nudging           = -6.0000
slope from the calculus rule 2(w - 3) = -6.0000   (the known rule for the slope of (w-3)^2)

nudge w up by 0.01: error goes 9.0000 -> 8.9401


**What to notice.** The slope is **negative** (-6), and nudging `w` *up* made the error go *down*. A negative slope means "more `w` is better". So the rule is: **move `w` opposite to the slope.** That's the whole algorithm:

`w_new = w - lr * slope`

`lr` is the **learning rate**: how big a step to take per unit of slope. It's the first knob. Watch eight steps with `lr = 0.1`:

In [3]:
w = 0.0
lr = 0.1           # WHAT: the learning rate. THE FIRST KNOB. It belongs to the SEARCH (how we hunt), not to the problem.

print(" step |    w    |  error  | slope")
for step in range(0, 9):
    s = slope_by_nudging(loss, w)
    print(f"  {step:2d}  | {w:7.4f} | {loss(w):7.4f} | {s:+.3f}")
    w = w - lr * s           # WHAT: the ONE line that is gradient descent: step opposite to the slope
                             # WHY:  the minus sign is "go downhill". A negative slope makes -lr*slope positive, so w increases.
                             # SAME AS: the Qmaze notebook's  new = old + alpha * (target - old)  (Part 8 proves it)

 step |    w    |  error  | slope
   0  |  0.0000 |  9.0000 | -6.000
   1  |  0.6000 |  5.7600 | -4.800
   2  |  1.0800 |  3.6864 | -3.840
   3  |  1.4640 |  2.3593 | -3.072
   4  |  1.7712 |  1.5099 | -2.458
   5  |  2.0170 |  0.9664 | -1.966
   6  |  2.2136 |  0.6185 | -1.573
   7  |  2.3709 |  0.3958 | -1.258
   8  |  2.4967 |  0.2533 | -1.007


**What to notice.** Two things happen as `w` closes in on 3. The error shrinks, and **the slope shrinks too**, so the steps get smaller automatically without anyone changing `lr`. The slope is steep far from the answer and flat near it, which is why descent slows down as it arrives.

# Part 2: The learning rate, and the rule that governs it

> **You are here:** you saw `lr = 0.1` work. What about other values?

**PREDICT first.** In the bowl above, start at `w = 0` and use `lr = 0.01`, `0.1`, `0.5`, `0.9`, `1.0` and `1.05`. For each one, guess: does it crawl toward 3, jump straight there, bounce back and forth, or run away? Type your guess below, *then* run the next cell.

In [4]:
# PREDICT: type your guess for each learning rate here (any words), then run the next cell.
my_prediction = ""
print("your prediction:", my_prediction if my_prediction else "(blank: type one above first. It is worth it.)")

your prediction: (blank: type one above first. It is worth it.)


In [5]:
def run_bowl(lr, steps=10, w0=0.0):
    w = w0
    for _ in range(steps):
        w = w - lr * slope_by_nudging(loss, w)
    return w

print("  lr  | error left after 10 steps | rule predicts | what happens")
for lr in [0.01, 0.1, 0.5, 0.9, 1.0, 1.05]:
    left = abs(run_bowl(lr) - 3.0)
    factor = 1 - 2 * lr                       # WHAT: the rule derived below. Each step multiplies the leftover error by this.
    predicted = 3.0 * abs(factor) ** 10
    if abs(factor) < 1e-9:   what = "jumps straight to the answer"
    elif abs(factor) < 1:    what = "converges" + (" (sign flips each step: zig-zag)" if factor < 0 else "")
    elif abs(factor) == 1:   what = "bounces forever, never settles"
    else:                    what = "RUNS AWAY"
    print(f" {lr:4} | {left:25.4g} | {predicted:13.4g} | {what}")

  lr  | error left after 10 steps | rule predicts | what happens
 0.01 |                     2.451 |         2.451 | converges
  0.1 |                    0.3221 |        0.3221 | converges
  0.5 |                         0 |             0 | jumps straight to the answer
  0.9 |                    0.3221 |        0.3221 | converges (sign flips each step: zig-zag)
  1.0 |                         3 |             3 | bounces forever, never settles
 1.05 |                     7.781 |         7.781 | RUNS AWAY


**Where the rule comes from (derive it once, by hand).** The slope of `(w - 3)^2` is `2(w - 3)`. One step is
`w_new = w - lr * 2(w - 3)`. Subtract 3 from both sides:

`w_new - 3 = (w - 3) - lr * 2 (w - 3) = (1 - 2*lr) * (w - 3)`

So **each step multiplies the leftover error by `(1 - 2*lr)`.** That one sentence explains the whole table:
- factor between 0 and 1 (`lr` below 0.5): smooth convergence. Smaller factor = faster.
- factor 0 (`lr = 0.5`): it lands on the answer in one step.
- factor between -1 and 0: it overshoots the answer each step but each overshoot is smaller (zig-zag, still converges).
- factor exactly -1 (`lr = 1.0`): it bounces between two points forever.
- factor beyond -1 (`lr` above 1): every overshoot is *bigger* than the last, so it runs away.

The two printed columns agree, so the rule is right, not just plausible. Here it is on a plot:

In [6]:
def error_path(lr, steps=15):
    w, errs = 0.0, [3.0]
    for _ in range(steps):
        w = w - lr * 2 * (w - 3.0)
        errs.append(abs(w - 3.0))
    return np.maximum(errs, 1e-12)          # (the floor only keeps the log axis from choking on a perfect 0)

fig, ax = plt.subplots(figsize=(7.2, 3.9))
for lr, color in [(0.1, BLUE), (0.4, ORANGE), (0.95, AQUA), (1.05, YELLOW)]:
    e = error_path(lr)
    ax.plot(e, color=color, label=f"lr = {lr}")
    tag(ax, len(e) - 1, e[-1], f"lr {lr}", color)
ax.set_yscale("log"); ax.set_xlabel("step"); ax.set_ylabel("distance from the answer, |w - 3|")
ax.set_title("Same bowl, four step sizes: slow, fast, zig-zag but shrinking, runaway")
ax.set_xlim(0, 17.5); ax.legend(loc="lower left")
plt.show()

<Figure size 792x429 with 1 Axes>

### The speed limit depends on how curved the bowl is
The bowl `(w - 3)^2` has a certain **stiffness** (curvature). A stiffer bowl `k/2 * (w - 3)^2` has slope `k(w - 3)`, and the same algebra gives a factor of `(1 - lr*k)`. So the speed limit is **`lr < 2/k`**: the stiffer the bowl, the smaller the step you can afford. Test the claim: just under the limit should converge, just over should run away.

In [7]:
print("  k  | limit 2/k |  lr (a bit under) -> error  |  lr (a bit over) -> error")
for k in (1, 10, 100):
    limit = 2.0 / k
    results = []
    for lr in (0.9 * limit, 1.1 * limit):
        w = 0.0
        for _ in range(100):
            w = w - lr * k * (w - 3.0)           # slope of (k/2)(w-3)^2 is k(w-3)
        results.append((lr, abs(w - 3.0)))
    print(f" {k:3d} | {limit:9.4f} |  {results[0][0]:.4f} -> {results[0][1]:<14.3g}|  {results[1][0]:.4f} -> {results[1][1]:.3g}")

  k  | limit 2/k |  lr (a bit under) -> error  |  lr (a bit over) -> error
   1 |    2.0000 |  1.8000 -> 6.11e-10      |  2.2000 -> 2.48e+08
  10 |    0.2000 |  0.1800 -> 6.11e-10      |  0.2200 -> 2.48e+08
 100 |    0.0200 |  0.0180 -> 6.11e-10      |  0.0220 -> 2.48e+08


**What to notice.** The limit moves a hundredfold between `k = 1` and `k = 100`, and the rule predicts it each time. **This is the key idea of the whole notebook: the best learning rate is not a property of the algorithm. It's set by the curvature of the problem.** Part 5 shows how that bites in real fitting.

# Part 3: Fitting a line (two parameters, many data points)

> **You are here:** one dial worked. Now a real fit: **two** parameters, and **many** data points.
> **New idea:** the loss is the *average* of squared misses, and there are now two slopes to measure, one per parameter.

The data: 40 measurements of a part. `x` is where you measured (think: distance from the center, -5 to +5) and `y` is the reading. The truth is `y = 2x + 1`, but each reading has noise. The model is a line `y_hat = m*x + b` with two parameters, slope `m` and intercept `b`.

**The loss** is the **mean squared error (MSE)**: for each point take the miss `(y_hat - y)`, square it, average. Squaring makes every miss positive and punishes big misses more. (That is a *choice*. Part 6 shows what a different choice does.)

In [8]:
rng = np.random.default_rng(0)
n = 40
x = np.linspace(-5, 5, n)
y = 2 * x + 1 + rng.normal(0, 1.5, n)             # WHAT: the truth (slope 2, intercept 1) plus random noise

def loss_line(p, X, Y):
    m, b = p
    return np.mean((m * X + b - Y) ** 2)          # WHAT: average of squared misses

def grad_line(p, X, Y):
    m, b = p
    miss = m * X + b - Y                           # WHAT: the miss for every data point (a list of 40 numbers)
    # WHY these formulas: change b by 1 and every miss shifts by 1, so  d(loss)/db = mean(2 * miss).
    #                     change m by 1 and miss number i shifts by x_i, so  d(loss)/dm = mean(2 * miss * x).
    #                     (that is the chain rule, said in words)
    return np.array([2 * np.mean(miss * X), 2 * np.mean(miss)])

# Never trust a formula you can check: compare it with nudge-and-measure on both parameters.
p_test = np.array([0.5, -0.3])
h = 1e-6
nudged = np.array([
    (loss_line(p_test + [h, 0], x, y) - loss_line(p_test - [h, 0], x, y)) / (2 * h),
    (loss_line(p_test + [0, h], x, y) - loss_line(p_test - [0, h], x, y)) / (2 * h),
])
print("slopes from my formulas :", np.round(grad_line(p_test, x, y), 5))
print("slopes by nudging       :", np.round(nudged, 5))

slopes from my formulas : [-27.09495  -2.41867]
slopes by nudging       : [-27.09495  -2.41867]


The two lists agree, so the formulas are right. (This "gradient check" is how people debug hand-written learning code, and we'll use it again when we build a neural network from scratch.) Now gradient descent, starting from the flat line `m = 0, b = 0`:

In [9]:
best_params = np.polyfit(x, y, 1)                 # WHAT: the exact answer from a textbook formula (we use it ONLY to grade the descent)
best_loss = loss_line(best_params, x, y)

p = np.array([0.0, 0.0])
lr = 0.05
history = [p.copy()]
losses = [loss_line(p, x, y)]
print(" step |    m     |    b     |   loss")
for step in range(1, 101):
    p = p - lr * grad_line(p, x, y)                # WHAT: the same one line as Part 1, now applied to BOTH parameters
    history.append(p.copy()); losses.append(loss_line(p, x, y))
    if step in (1, 2, 5, 10, 20, 50, 100):
        print(f"  {step:3d} | {p[0]:8.4f} | {p[1]:8.4f} | {losses[-1]:8.4f}")
print()
print(f"textbook answer: m = {best_params[0]:.4f}, b = {best_params[1]:.4f}, loss = {best_loss:.4f}   (the truth was m = 2, b = 1; the noise moves the fit a little)")

 step |    m     |    b     |   loss
    1 |   1.7928 |   0.0909 |   2.6084
    2 |   2.0150 |   0.1728 |   1.9263
    5 |   2.0463 |   0.3724 |   1.6635
   10 |   2.0464 |   0.5923 |   1.4757
   20 |   2.0464 |   0.7988 |   1.3874
   50 |   2.0464 |   0.9046 |   1.3752
  100 |   2.0464 |   0.9093 |   1.3752

textbook answer: m = 2.0464, b = 0.9093, loss = 1.3752   (the truth was m = 2, b = 1; the noise moves the fit a little)


In [10]:
history = np.array(history)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.9))
a1.scatter(x, y, s=18, color=MUTED, zorder=3)
xs = np.array([-5, 5])
for step, color in zip((0, 1, 5, 50), BLUE_RAMP):
    m_, b_ = history[step]
    a1.plot(xs, m_ * xs + b_, color=color)
    tag(a1, 5, m_ * 5 + b_, "start" if step == 0 else f"step {step}", color)
a1.set_xlim(-5.5, 7.5); a1.set_xlabel("x"); a1.set_ylabel("y"); a1.set_title("The line swings into place (darker = later)")
a2.plot(np.array(losses) - best_loss + 1e-9, color=BLUE)
a2.set_yscale("log"); a2.set_xlabel("step"); a2.set_ylabel("loss above the best possible")
a2.set_title("Loss falls fast, then slowly (log scale)")
plt.tight_layout(); plt.show()

<Figure size 1210x429 with 2 Axes>

**What to notice.**
- Look at the table: the slope `m` is nearly right after 2 steps, but the intercept `b` takes about 50. The two parameters don't move at the same speed. Hold that thought for Part 5.
- For a line there is a formula that gives the answer directly. For a neural network with millions of parameters **there isn't one**, which is why we need a search rule like this. Descent is how we find an answer when no formula exists.
- Variants of this loop (with names like Adam) train the Keras networks from the earlier notebooks.

# Part 4: The learning-rate speed limit, for the line fit

> **You are here:** Part 2 showed the speed limit in a one-parameter bowl. Does the same idea hold for the two-parameter fit?

With two parameters, the bowl can be curved differently in different directions, so its curvature is a 2x2 table of numbers (the "slopes of the slopes"). Its **largest** value, call it `lambda_max`, is the stiffest direction, and it sets the speed limit: **`lr < 2 / lambda_max`**. Compute it, then test it with a sweep.

In [11]:
def curvature_matrix(X):
    # WHAT: the 2x2 "stiffness" table of the MSE bowl for a line fit (second derivatives of the loss).
    # WHY:  its biggest eigenvalue is the stiffest direction. We just hand you the matrix; it comes from differentiating grad_line once more.
    return np.array([[2 * np.mean(X ** 2), 2 * np.mean(X)],
                     [2 * np.mean(X),      2.0]])

eig = np.linalg.eigvalsh(curvature_matrix(x))        # the stiffness along the bowl's softest and stiffest directions
limit_centered = 2.0 / eig.max()
print(f"stiffness of the two directions: soft = {eig.min():.2f}, stiff = {eig.max():.2f}")
print(f"predicted speed limit: lr < 2 / {eig.max():.2f} = {limit_centered:.4f}")

def steps_to_tol(lr, X, Y, best_p, cap=5000, tol=1e-4):
    # WHAT: run plain descent from (0, 0) until the loss is within `tol` of the best possible. Report how many steps it took.
    # WHY:  the ruler for this notebook. "How many steps" is the cost of a choice of lr.
    p = np.zeros(2); target_loss = loss_line(best_p, X, Y)
    for t in range(1, cap + 1):
        p = p - lr * grad_line(p, X, Y)
        l = loss_line(p, X, Y)
        if not np.isfinite(l) or l > 1e12:
            return "blew up"
        if l - target_loss < tol:
            return t
    return f">{cap}"

print()
print("  lr   | steps to get within 1e-4 of the best loss")
sweep = {}
for lr in [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.4]:
    sweep[lr] = steps_to_tol(lr, x, y, best_params)
    print(f" {lr:5} | {sweep[lr]}")

stiffness of the two directions: soft = 2.00, stiff = 17.52
predicted speed limit: lr < 2 / 17.52 = 0.1141

  lr   | steps to get within 1e-4 of the best loss
 0.001 | 2253
 0.005 | 449
  0.01 | 224
  0.05 | 43
   0.1 | 24
   0.2 | blew up
   0.4 | blew up


In [12]:
lrs = np.logspace(-3, np.log10(0.2), 16)
fig, ax = plt.subplots(figsize=(7.2, 3.9))
ok_lr, ok_steps, bad_lr = [], [], []
for lr in lrs:
    r = steps_to_tol(lr, x, y, best_params)
    if isinstance(r, int): ok_lr.append(lr); ok_steps.append(r)
    else: bad_lr.append(lr)
ax.plot(ok_lr, ok_steps, color=BLUE, marker="o", markersize=4)
ax.axvline(limit_centered, color=ORANGE, linestyle="--", linewidth=1.4)
tag(ax, limit_centered, ax.get_ylim()[1] * 0.6, "predicted speed limit", ORANGE, dx=-4, ha="right")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("learning rate"); ax.set_ylabel("steps needed")
ax.set_title("Bigger steps are faster, right up to the speed limit, then it blows up")
plt.show()
print("learning rates in this sweep that blew up (to the right of the dashed line):", [round(float(v), 3) for v in bad_lr])

<Figure size 792x429 with 1 Axes>

learning rates in this sweep that blew up (to the right of the dashed line): [0.14, 0.2]


**What to notice.**
- Steps needed fall almost in proportion as `lr` grows. Ten times the step size needs about a tenth of the steps, **until you cross the predicted limit**, where it blows up. The line sits between the last learning rate that worked and the first that failed.
- So there is a **window**: too small wastes steps, too big explodes, and the window's upper edge is computed from the problem's curvature.
- That's why `lr = 0.1` in Part 3 felt "fine" while `0.2` would have broken. Neither number is special. The curvature decides.

# Part 5: The units trap (why the same problem can be 10x slower)

> **You are here:** you know the speed limit is set by the problem's curvature. Now see what changes the curvature without changing the problem at all.

Take exactly the same 40 measurements, and just **measure `x` from a different datum**: from the left edge (0 to 10) instead of from the center (-5 to +5). Same parts, same readings, same best-fit line. Only the numbers describing `x` shifted by 5.

**PREDICT:** will descent need about the same number of steps, fewer, or more? Type a guess, then run.

In [13]:
# PREDICT: same data, x shifted by 5. Same steps, fewer, or more? And by roughly how much?
my_prediction = ""
print("your prediction:", my_prediction if my_prediction else "(blank: type one above first)")

your prediction: (blank: type one above first)


In [14]:
x_shift = x + 5.0                                   # WHAT: the SAME data measured from a different datum (0..10 instead of -5..5)
best_shift = np.polyfit(x_shift, y, 1)

eig_s = np.linalg.eigvalsh(curvature_matrix(x_shift))
limit_shift = 2.0 / eig_s.max()
print(f"centered x : stiffness soft/stiff = {eig.min():6.2f} / {eig.max():6.2f}   ratio {eig.max()/eig.min():6.1f}   speed limit {limit_centered:.4f}")
print(f"shifted  x : stiffness soft/stiff = {eig_s.min():6.2f} / {eig_s.max():6.2f}   ratio {eig_s.max()/eig_s.min():6.1f}   speed limit {limit_shift:.4f}")
print()
lr_c, lr_s = 0.9 * limit_centered, 0.9 * limit_shift      # WHAT: give each version the best safe step: 90% of ITS OWN speed limit
steps_centered = steps_to_tol(lr_c, x, y, best_params)
steps_shifted = steps_to_tol(lr_s, x_shift, y, best_shift)
print(f"centered x at lr = {lr_c:.4f}: {steps_centered} steps")
print(f"shifted  x at lr = {lr_s:.4f}: {steps_shifted} steps")
print()
# Same line? Predict y at x = 3 (measured the centered way; 8 in the shifted way) with each fit.
print("same fitted line?  prediction at x = 3 (centered) :", round(best_params[0] * 3 + best_params[1], 4),
      "| at x = 8 (shifted):", round(best_shift[0] * 8 + best_shift[1], 4))

centered x : stiffness soft/stiff =   2.00 /  17.52   ratio    8.8   speed limit 0.1141
shifted  x : stiffness soft/stiff =   0.51 /  69.01   ratio  135.9   speed limit 0.0290

centered x at lr = 0.1027: 29 steps
shifted  x at lr = 0.0261: 464 steps

same fitted line?  prediction at x = 3 (centered) : 7.0485 | at x = 8 (shifted): 7.0485


**Why it's slower.** Both versions have a stiffest direction that sets the speed limit, but also a **softest** direction, and the softest direction sets the *pace* (the slow direction improves by a factor of roughly `1 - lr * soft` per step). When `soft` is tiny next to `stiff`, you must take small steps (because of stiff) and you move slowly along soft. The **ratio** of stiff to soft is the problem. Shifting `x` made that ratio far worse. Here are the two loss surfaces, with the path of descent drawn on each:

In [15]:
def gd_path(lr, X, Y, steps=40):
    p = np.zeros(2); path = [p.copy()]
    for _ in range(steps):
        p = p - lr * grad_line(p, X, Y); path.append(p.copy())
    return np.array(path)

def surface(X, Y, m_rng, b_rng, n=160):
    M, B = np.meshgrid(np.linspace(*m_rng, n), np.linspace(*b_rng, n))
    Z = np.mean((M[..., None] * X + B[..., None] - Y) ** 2, axis=-1)
    return M, B, Z

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, X, best_p, lr_, b_rng, title in [
        (axes[0], x, best_params, 0.5 * limit_centered, (-3.5, 3.5), "x measured from the center"),
        (axes[1], x_shift, best_shift, 0.5 * limit_shift, (-14, 3.5), "x measured from the edge")]:
    M, B, Z = surface(X, y, (-0.8, 3.6), b_rng)
    best_l = loss_line(best_p, X, y)
    ax.contour(M, B, Z, levels=best_l + np.array([0.05, 0.3, 1, 3, 10, 30, 100, 300]), colors="#898781", linewidths=0.7)
    path = gd_path(lr_, X, y, 40)
    ax.plot(path[:, 0], path[:, 1], color=BLUE, marker="o", markersize=3, linewidth=1.4)
    ax.scatter([best_p[0]], [best_p[1]], color=ORANGE, zorder=5, s=36)
    ax.set_xlabel("slope m"); ax.set_ylabel("intercept b"); ax.set_title(title + "  (first 40 steps, lr at half its limit)")
    tag(ax, best_p[0], best_p[1], "best fit", ORANGE, dx=6, dy=8)
plt.tight_layout(); plt.show()

<Figure size 1210x462 with 2 Axes>

**What to notice.** The rings are lines of equal loss. On the left they are close to round, so the path heads almost straight to the answer. On the right they are a long, narrow, tilted valley: the path **zig-zags across the narrow direction** (the stiff direction is what forces small steps) and then creeps slowly **along** the valley floor, still far from the best fit after 40 steps. Same problem, different coordinates, many more steps.

**Fixes, from simplest.**
1. **Change the coordinates:** center (and scale) the inputs. This is called *standardizing*. It's a choice about how you present the data, not about the algorithm.
2. **Change the search rule:** momentum (give the ball mass so it rolls along the valley) or adaptive step sizes. You'll race these in the sandbox in Part 9.

Try fix 1 right now: standardize `x_shift` (subtract its mean, divide by its standard deviation) and see.

In [16]:
mu, sd = x_shift.mean(), x_shift.std()
x_std = (x_shift - mu) / sd                        # WHAT: center on 0 and scale so the typical spread is 1
best_std = np.polyfit(x_std, y, 1)
eig_std = np.linalg.eigvalsh(curvature_matrix(x_std))
limit_std = 2.0 / eig_std.max()
steps_standardized = steps_to_tol(0.9 * limit_std, x_std, y, best_std)

print(f"standardized x: stiffness soft/stiff = {eig_std.min():.2f} / {eig_std.max():.2f}  (ratio {eig_std.max()/eig_std.min():.1f})")
print(f"steps at 90% of its speed limit: {steps_standardized}   (centered took {steps_centered}, shifted took {steps_shifted})")
x_new = 8.0                                        # a point measured the shifted way
print("same line still?  prediction at x = 8 from the standardized fit:",
      round(best_std[0] * ((x_new - mu) / sd) + best_std[1], 4), "| from the shifted fit:", round(best_shift[0] * x_new + best_shift[1], 4))

standardized x: stiffness soft/stiff = 2.00 / 2.00  (ratio 1.0)
steps at 90% of its speed limit: 29   (centered took 29, shifted took 464)
same line still?  prediction at x = 8 from the standardized fit: 7.0485 | from the shifted fit: 7.0485


**What to notice.** The line it finds is the same line (the two predictions at `x = 8` match). Only the coordinates changed, and the step count dropped by a large factor. **This is why nearly every ML pipeline rescales its inputs before training**, and a whole class of "why won't my model train" problems is really this.

# Part 6: The loss is a choice

> **You are here:** you've been using squared error without questioning it. It's one choice among many, and the choice decides what "best" means.

Example with no descent needed: one number `c` that should summarize the data `[1, 2, 3, 4, 100]`. The last reading is a bad measurement (chatter on the gauge). Search every possible `c` and see which minimizes each loss.

In [17]:
data = np.array([1, 2, 3, 4, 100.0])
grid = np.linspace(0, 110, 11001)                                   # WHAT: try every c from 0 to 110 in steps of 0.01
mse_curve = np.array([np.mean((data - c) ** 2) for c in grid])      # WHAT: loss A: average SQUARED miss
mae_curve = np.array([np.mean(np.abs(data - c)) for c in grid])     # WHAT: loss B: average ABSOLUTE miss

best_c_mse = grid[np.argmin(mse_curve)]
best_c_mae = grid[np.argmin(mae_curve)]
print(f"best c under squared error : {best_c_mse:6.2f}    (the mean of the data = {data.mean():.2f})")
print(f"best c under absolute error: {best_c_mae:6.2f}    (the median of the data = {np.median(data):.2f})")

best c under squared error :  22.00    (the mean of the data = 22.00)
best c under absolute error:   3.00    (the median of the data = 3.00)


**What to notice.** Squared error picks the **mean** and absolute error picks the **median**. Squaring a miss of 96 gives 9,216, so the single bad reading dominates and drags the answer to 22. Absolute error counts that same miss as 96, so the outlier gets one vote, and the answer stays near the good data.

Neither loss is "correct". They answer different questions. If a bad reading is a mistake, absolute error (or other robust losses) is the more durable choice. If big misses really are much more costly (a part out of tolerance), squared error is right. **The loss is where you tell the algorithm what you care about**, and it's a knob in the *objective* family.

# Part 7: Mini-batches (the batch-size knob)

> **You are here:** every step so far used **all 40** data points to compute the slope. With a million points, that's slow.

The shortcut: compute the slope from a random **handful** of points. It's noisier, but much cheaper per step. How many points per step is the **batch size**. The Q-learning update used a batch of **1** (one move), and the DQN notebook used batches of 32. Same knob.

In [18]:
def sgd_losses(batch_size, lr=0.02, updates=200, seed=0):
    r = np.random.default_rng(seed)
    p = np.zeros(2); out = []
    for _ in range(updates):
        idx = r.choice(n, batch_size, replace=False) if batch_size < n else np.arange(n)   # WHAT: pick this step's handful of points
        p = p - lr * grad_line(p, x[idx], y[idx])                                          # WHAT: the SAME update, slope from just those points
        out.append(loss_line(p, x, y) - best_loss)                                         # WHAT: but we JUDGE it on all 40 points
    return np.array(out)

fig, ax = plt.subplots(figsize=(7.2, 3.9))
batch_gaps = {}
for bs, color in [(40, BLUE), (8, ORANGE), (1, AQUA)]:
    l = sgd_losses(bs)
    ax.plot(np.maximum(l, 1e-7), color=color, label=f"batch size {bs}")
    tag(ax, len(l) - 1, max(l[-1], 1e-7), f"batch {bs}", color)
    gaps = [sgd_losses(bs, seed=s)[-1] for s in range(5)]
    batch_gaps[bs] = float(np.mean(gaps))
ax.set_yscale("log"); ax.set_xlabel("update"); ax.set_ylabel("loss above the best possible (all 40 points)")
ax.set_xlim(0, 235); ax.set_title("Smaller batches: cheaper steps, but they never quite settle")
ax.legend(loc="lower left")
plt.show()
print("average final loss gap over 5 random seeds (smaller = closer to the best fit):")
for bs, g in batch_gaps.items():
    print(f"   batch size {bs:2d}: {g:.5f}")

<Figure size 792x429 with 1 Axes>

average final loss gap over 5 random seeds (smaller = closer to the best fit):
   batch size 40: 0.00000
   batch size  8: 0.00416
   batch size  1: 0.19630


**What to notice.** With all 40 points the loss keeps falling smoothly. With a batch of 8 it settles higher and jitters. With a batch of 1 it **jumps around a noise floor** and doesn't settle at a fixed learning rate. That's the same trade as `alpha` in the slippery maze: a noisy single observation can mislead you, so you either average over a batch or shrink the step. Batch size and learning rate are linked knobs: noisier slopes need smaller steps.

# Part 8: The bridge back to the Qmaze update

> **You are here:** the reason this notebook comes first. Here's the claim: the update rule you've been reading in Q-learning **is a gradient-descent step**.

In the Qmaze notebook the table update was `new = old + alpha * (target - old)`. Treat the target as a fixed number and define the loss `0.5 * (target - q)^2`, "how far off is my guess `q` from the target". Take one descent step on `q` with learning rate `alpha`. Check it numerically with the Part 1 tools, using the pretend numbers from the Qmaze notebook's hand arithmetic:

In [19]:
alpha, old_q, target = 0.5, 0.0, 0.68             # the pretend numbers from the Qmaze notebook (target = -0.04 + 0.9 * 0.8)

def guess_loss(q):
    # WHAT: "how wrong is my guess q?" with the target treated as a FIXED number.
    # WATCH: that "treated as fixed" is exactly what `tf.stop_gradient(targets - values)` did in the A2C code: don't let the
    #        search move the answer key, only the guess.
    return 0.5 * (target - q) ** 2

slope = slope_by_nudging(guess_loss, old_q)                  # the slope at the old guess, by nudging
descent_step = old_q - alpha * slope                         # Part 1's update:  w - lr * slope
q_learning_step = old_q + alpha * (target - old_q)           # the Qmaze notebook's update

print(f"slope of the loss at q = {old_q}: {slope:+.4f}   (calculus rule: -(target - q) = {-(target - old_q):+.4f})")
print(f"one gradient-descent step with lr = alpha : {descent_step:.4f}")
print(f"the Q-learning update                     : {q_learning_step:.4f}")

slope of the loss at q = 0.0: -0.6800   (calculus rule: -(target - q) = -0.6800)
one gradient-descent step with lr = alpha : 0.3400
the Q-learning update                     : 0.3400


**What to notice.** The two numbers are the same, because the slope of `0.5*(target - q)^2` is `-(target - q)`, so `q - alpha * slope = q + alpha*(target - q)`. This shows three things:
- **`alpha` in Q-learning is a learning rate.** Everything you learned in Parts 2 to 5 applies. Too big and the estimates can run away (Part 2), and the right size depends on the curvature of the problem.
- **A Q-table update is gradient descent on ONE parameter**, the single cell being changed. A Keras network does the same thing to **every weight at once**, each moved by its own slope. That's what `train_on_batch` did.
- **`gamma` and `epsilon` don't appear here.** They belong to the reinforcement-learning layer (what the target means, and how to gather experience), not to the descent machine. Knowing which layer each knob lives in is half of tuning.

# Part 9: Sandbox (your ideas compete here)

> **You are here:** you've seen three problems that hurt differently: a nicely-scaled bowl, a long thin valley, and a very stiff bowl. Plain descent needs a different learning rate for each. Can a smarter *search rule* handle all three with no hand-tuning?

**The rules of the ruler.** An optimizer is a function `opt(grad, loss, w0, max_steps)` that returns the path of weights it visited. It can only call `grad(w)` and `loss(w)`. It is scored by **how many steps until the loss is within 1e-4 of the best possible**, up to 5,000 steps. Each problem is run from a fixed start.

The reference entries are standard ideas:
- **Plain descent** with a fixed learning rate (what we've used so far).
- **Momentum** (the heavy ball): keep a velocity, `v = beta*v - lr*slope`, then `w = w + v`. It builds up speed along a consistent direction and damps the zig-zag.
- **Bold driver** (a simple adaptive step): if a step lowers the loss, accept it and grow `lr` by 10%. If it raises the loss, reject it and halve `lr`. No `lr` to pick.

In [20]:
# The three problems. Each has a loss, a gradient, a starting point and the best possible loss.
def make_line_problem(X, Y, name):
    A = np.column_stack([X, np.ones_like(X)])
    best_p = np.linalg.lstsq(A, Y, rcond=None)[0]
    return dict(name=name,
                loss=lambda p: loss_line(p, X, Y),
                grad=lambda p: grad_line(p, X, Y),
                w0=[0.0, 0.0],
                best=loss_line(best_p, X, Y))

problems = [
    make_line_problem(x, y, "line, centered"),
    make_line_problem(x_shift, y, "line, shifted"),
    dict(name="stiff bowl", loss=lambda w: 0.5 * (w[0] ** 2 + 100 * w[1] ** 2),
         grad=lambda w: np.array([w[0], 100 * w[1]]), w0=[10.0, 10.0], best=0.0),
]

def plain_gd(lr):
    def run(grad, loss, w0, max_steps):
        w = np.array(w0, float); path = [w.copy()]
        for _ in range(max_steps):
            w = w - lr * grad(w); path.append(w.copy())
        return np.array(path)
    return run

def momentum(lr, beta):
    def run(grad, loss, w0, max_steps):
        w = np.array(w0, float); v = np.zeros_like(w); path = [w.copy()]
        for _ in range(max_steps):
            v = beta * v - lr * grad(w)            # WHAT: velocity remembers the previous steps (a ball with mass)
            w = w + v
            path.append(w.copy())
        return np.array(path)
    return run

def bold_driver(lr0=1.0):
    def run(grad, loss, w0, max_steps):
        w = np.array(w0, float); lr = lr0; cur = loss(w); path = [w.copy()]
        for _ in range(max_steps):
            cand = w - lr * grad(w); lc = loss(cand)
            if np.isfinite(lc) and lc <= cur:
                w, cur = cand, lc; lr *= 1.1       # WHAT: the step helped: accept it and be a little bolder
            else:
                lr *= 0.5                          # WHAT: the step hurt: reject it and be more careful
            path.append(w.copy())
        return np.array(path)
    return run

def race(entries, problems, max_steps=5000, tol=1e-4):
    # WHAT: the ruler. Run every entry on every problem and report steps-to-tolerance (or "blew up" / "no").
    names = [P["name"] for P in problems]
    print(f"{'':30s}" + "".join(f"{nm:>16s}" for nm in names))
    results = {}
    for label, opt in entries:
        row = []
        for P in problems:
            try:
                with np.errstate(all="ignore"):
                    path = opt(P["grad"], P["loss"], P["w0"], max_steps)
                    gaps = np.array([P["loss"](w) for w in path]) - P["best"]
                if not np.all(np.isfinite(gaps)) or np.max(np.abs(gaps)) > 1e12:
                    row.append("blew up")
                elif np.any(gaps < tol):
                    row.append(str(int(np.argmax(gaps < tol))))
                else:
                    row.append(f"no (>{max_steps})")
            except Exception as err:
                row.append("error: " + str(err)[:20])
        results[label] = row
        print(f"{label:30s}" + "".join(f"{r:>16s}" for r in row))
    return results

reference = [
    ("plain, lr = 0.01 (safe)", plain_gd(0.01)),
    ("plain, lr = 0.1", plain_gd(0.1)),
    ("momentum, lr 0.02, beta 0.8", momentum(0.02, 0.8)),
    ("bold driver (no lr to pick)", bold_driver()),
]
race_results = race(reference, problems)

                                line, centered   line, shifted      stiff bowl


plain, lr = 0.01 (safe)                    224            1213             653


plain, lr = 0.1                             24         blew up         blew up


momentum, lr 0.02, beta 0.8                 47              89              75


bold driver (no lr to pick)                 26             451             361


**What to notice.**
- **No single fixed learning rate wins everywhere.** `lr = 0.1` is fast on the first problem and blows up on the other two. `lr = 0.01` is safe but slow on all of them.
- Momentum is the only entry that is reasonable on all three problems *with the settings I picked* (it loses to plain `lr = 0.1` and the bold driver on the first problem, where those are already well-suited). I tried a few settings in a quick test and kept one, so this is not a fair ranking of methods. It shows what momentum can do, not that it always wins.
- The bold driver needs **no hand-tuned learning rate**. It finds the speed limit itself. It still pays for the valley's bad shape on the shifted problem, because that's a *conditioning* problem, which only a different search rule (momentum) or different coordinates (standardizing) fixes.

### Your slot
Replace the body of `my_optimizer` with your own idea and re-run the next cell. Ideas to try: a different momentum `beta`; a learning rate that decays as `1/(1 + step/100)`; a separate learning rate for each parameter; "if the last two steps point in opposite directions, shrink the learning rate"; or something of your own. It plays against the same ruler and the same reference entries.

In [21]:
def my_optimizer(grad, loss, w0, max_steps):
    # SANDBOX: replace this body with your own idea. It receives grad(w) and loss(w) and must return the path of weights.
    # It starts as plain descent so the cell runs. See how a change moves the numbers in the last row.
    w = np.array(w0, float); path = [w.copy()]
    lr = 0.05
    for step in range(max_steps):
        w = w - lr * grad(w)
        path.append(w.copy())
    return np.array(path)

_ = race(reference + [("MY OPTIMIZER", my_optimizer)], problems)

                                line, centered   line, shifted      stiff bowl


plain, lr = 0.01 (safe)                    224            1213             653


plain, lr = 0.1                             24         blew up         blew up


momentum, lr 0.02, beta 0.8                 47              89              75


bold driver (no lr to pick)                 26             451             361


MY OPTIMIZER                                43         blew up         blew up


# Part 10: The knob ledger, self-check, and what's next

## Knob ledger (Notebook 1)
Every knob belongs to one of five families: **objective** (what you're optimizing), **model** (what's inside), **search** (how you hunt), **experience** (what data/trials you feed it), **judging** (when you stop and how you score). The numbers below are computed from this notebook's own runs.

In [22]:
rows = [
    ("learning rate", "search",
     f"too small: {sweep[0.001]} steps at lr 0.001",
     f"too big: blows up above ~{limit_centered:.3f} (= 2/stiffness)"),
    ("momentum beta", "search",
     "beta = 0 is plain descent (no help in a valley)",
     "beta near 1 builds up so much speed it can overshoot (not tested here: try it in the sandbox)"),
    ("batch size", "search",
     f"batch 1: final loss gap {batch_gaps[1]:.4f} (jumpy)",
     f"batch 40: gap {batch_gaps[40]:.5f} (smooth, but costs all the data every step)"),
    ("choice of loss", "objective",
     f"squared error answers {best_c_mse:.1f} (the mean) on the outlier data",
     f"absolute error answers {best_c_mae:.1f} (the median)"),
    ("input centering / scaling", "experience",
     f"shifted x: {steps_shifted} steps at its best lr",
     f"standardized x: {steps_standardized} steps"),
    ("stopping rule (tolerance)", "judging",
     "stop too early: the fit is not there yet",
     "stop too late: you pay for steps that no longer help"),
]
print(f"{'knob':28s}{'family':13s}{'at one extreme':52s}{'at the other'}")
print("-" * 140)
for knob, fam, a, b in rows:
    print(f"{knob:28s}{fam:13s}{a:52s}{b}")

knob                        family       at one extreme                                      at the other
--------------------------------------------------------------------------------------------------------------------------------------------
learning rate               search       too small: 2253 steps at lr 0.001                   too big: blows up above ~0.114 (= 2/stiffness)
momentum beta               search       beta = 0 is plain descent (no help in a valley)     beta near 1 builds up so much speed it can overshoot (not tested here: try it in the sandbox)
batch size                  search       batch 1: final loss gap 0.1963 (jumpy)              batch 40: gap 0.00000 (smooth, but costs all the data every step)
choice of loss              objective    squared error answers 22.0 (the mean) on the outlier dataabsolute error answers 3.0 (the median)
input centering / scaling   experience   shifted x: 464 steps at its best lr                 standardized x: 29 steps
stopping ru

## Self-check (answer in your own words before you read the answers)
| Question | Short answer |
|---|---|
| Why is there a **minus** sign in `w - lr * slope`? | The slope points uphill (the direction the error grows). We want to go downhill, so we move opposite. |
| In the bowl, what multiplies the leftover error each step? | `1 - lr * k` (`k` = the bowl's stiffness). Between -1 and 1 it converges; beyond that it runs away. |
| Why did `lr = 0.2` blow up on the line fit but `0.1` didn't? | The speed limit is `2 / (stiffest curvature)`, about 0.114 for that data. |
| Why was the shifted-`x` version so much slower? | The loss valley became long and thin: very different stiffness in two directions. The stiff direction caps the step; the soft direction sets the pace. |
| What is `alpha` in Q-learning, in gradient-descent terms? | The learning rate for a descent step on `0.5 * (target - q)^2`, treating the target as fixed. |
| Which of squared or absolute error is more robust to a bad reading? | Absolute error: it gives the outlier one vote (median), where squaring gives it a huge one (mean). |

## What you have now, and what's next
- You can say exactly what "learning" is: **parameters, a loss, and a search**.
- You know the update behind `alpha`, `lr`, and `train_on_batch`, and why its main knob has a computable limit.
- You've met five knob families and the trade-offs at each end.

**Notebook 2: one neuron.** Same machine, new loss: instead of fitting a line to numbers, we classify (pass / fail) with **logistic regression**, which is a single neuron. That's the unit a neural network is built from, and the ledger will gain its first *model* knobs.